# Academic Failure Prediction Model

### Goal:
- use the given dataset about students and past exams to build a ML model to predict students grades.
- students with predicted scores under 50% will then receive additional support from School Management.


###  Data Pre-processing (10%)
- Missing values / outliers : Robust and nuanced approach
- Categorical encoding : Advanced or balanced encoding strategies
- Scaling numerical features : Method adapted to model needs

### Feature Selection and Creation (5%)
- Relevant feature selection : Justified selection using data or domain logic Optimal features chosen via analysis
- Feature transformation / creation (Optional depending on the necessity) : Useful engineered features Innovative and insightful feature creation.

### Model Comparison and Hyperparameter Selection (25%)
- Dataset splitting : train/val/test split and crossvalidation
- Model training : more than 3 models tested (among which one deep learning model) with choice justification.
- Performance comparison : In-depth multi-metric evaluation
- Hyperparameter selection/tuning : Advanced tuning with crossvalidation
- Over/underfitting analysis : Correct analysis with remedies, Deep, justified understanding

### Presentation of Results (20%)




### Report :
- EDA requires multiple graphs to be complete.
- When evaluating your failure prediction model, use a baseline (for example, the naive average of the students’ grades). This allows you to compare your model with a very simple ”worst-case” model and determine whether your approach performs better.
- For evaluating the OCR system, consider using a confusion matrix.


### Setup & Imports:

We use a kaggle jupyter server.

In [7]:
# from learntools.core import binder
# binder.bind(globals())


# from learntools.machine_learning.ex2 import *
# from learntools.feature_engineering_new.ex2 import *
# from learntools.ml_intermediate.ex1 import *

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Machine Learning
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor



# Feature Engineering
from sklearn.feature_selection import mutual_info_regression



print("Setup Complete")

Setup Complete


### Read Data:

In [ ]:
import pandas as pd
from pathlib import Path


# Path of the file to read
dataset_file_path = 'data/project_2026_dataset/student_dataset/student_failure/train.csv'

# # Read the file into a variable
raw_data = pd.read_csv(dataset_file_path) 

# Clean data
raw_data = raw_data.dropna(axis=0)

# Print
print(raw_data.columns)
raw_data.describe()


Index(['id', 'age', 'genre', 'diplôme', 'heures_etude', 'assiduité_classe',
       'accès_internet', 'heures_sommeil', 'qualité_sommeil', 'méthode_etude',
       'évaluation_établissement', 'difficulté_examen', 'score_examen',
       'heures_fête', 'taille_etudiant'],
      dtype='str')


,id,age,heures_etude,assiduité_classe,heures_sommeil,score_examen,heures_fête,taille_etudiant
count,630000.000000,630000.000000,611100.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,20.545821,4.002779,71.987261,7.072758,63.406789,4.000514,1.754836
std,181865.479132,2.260238,2.359388,17.430098,1.744811,18.950840,2.000382,0.131891
min,0.000000,17.000000,0.080000,40.600000,4.100000,12.600000,0.000000,1.480000
25%,157499.750000,19.000000,1.980000,57.000000,5.600000,49.600000,3.000000,1.660000
50%,314999.500000,21.000000,4.010000,72.600000,7.100000,63.620000,4.000000,1.750000
75%,472499.250000,23.000000,6.050000,87.200000,8.600000,77.300000,5.000000,1.850000
max,629999.000000,24.000000,7.910000,99.400000,9.900000,100.000000,19.000000,2.030000


EDA: target distribution, missingness, category plots, correlations
Feature Selection/Creation: mutual information, engineered features, dropped leakage-prone fields
Preprocessing: imputation, one-hot encoding, scaling, outlier handling
Baseline: naive mean predictor or pass/fail majority class
Models: at least 3; I’d suggest linear model, random forest, XGBoost, plus a small neural net if you want the advanced bonus
Validation: cross-validation, test split, MAE/RMSE or Accuracy/Precision/Recall/F1
Interpretation: feature importance + one example explanation

### Basic model

In [ ]:
all_feature_names = ['id', 'age', 'genre', 'diplôme', 'heures_etude', 'assiduité_classe',
       'accès_internet', 'heures_sommeil', 'qualité_sommeil', 'méthode_etude',
       'évaluation_établissement', 'difficulté_examen', 'score_examen',
       'heures_fête', 'taille_etudiant']

# Note : target should NOT be inside X
chosen_feature_names = ['id', 'age', 'genre', 'diplôme', 'heures_etude', 'assiduité_classe',
       'accès_internet', 'heures_sommeil', 'qualité_sommeil', 'méthode_etude',
       'évaluation_établissement', 'difficulté_examen',
       'heures_fête', 'taille_etudiant']

# Select data corresponding to features in feature_names
X = raw_data[chosen_feature_names]

# Convert categorical columns to numeric with one-hot encoding
X = pd.get_dummies(X)

# Prediction target
y = raw_data['score_examen']

# Split data
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)

# Specify and fit the model
dtr_model = DecisionTreeRegressor(random_state=2)
dtr_model.fit(train_X, train_y)

# Make predictions
predictions = dtr_model.predict(val_X)

#Compute MEA
val_mae = mean_absolute_error(val_y, predictions)


print(predictions)
print(val_mae)


[76.9  65.2  79.9  ... 54.5  38.14 76.6 ]
10.605811263492065


In [ ]:
# Analyze Under and overfitting
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

# compare MAE with differing values of max_leaf_nodes
for max_leaf_nodes in [5, 50, 500, 5000]:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print("Max leaf nodes: %d  \t\t Mean Absolute Error:  %d" %(max_leaf_nodes, my_mae))


Max leaf nodes: 5  		 Mean Absolute Error:  10
Max leaf nodes: 50  		 Mean Absolute Error:  8
Max leaf nodes: 500  		 Mean Absolute Error:  7
Max leaf nodes: 5000  		 Mean Absolute Error:  7


In [13]:
# Using Random Forests
# -> random forest uses many trees, and it makes a prediction by averaging the predictions of each component tree. It generally has much better predictive accuracy than a single decision tree and it works well with default parameters. If you keep modeling, you can learn more models with even better performance, but many of those are sensitive to getting the right parameters.



forest_model = RandomForestRegressor(random_state=1)
forest_model.fit(train_X, train_y)
RF_preds = forest_model.predict(val_X)
print(mean_absolute_error(val_y, RF_preds))



7.483691529523809


## Outliers and Missing Values



1. Drop Columns with Missing Values
2. Imputation
3. Imputation + Flag




In [ ]:
# Divide data into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    train_size=0.8, 
    test_size=0.2, 
    random_state=0
)

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)



#  Approach 1 (Drop Columns with Missing Values)

# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns
                     if X_train[col].isnull().any()]

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))




---
## Advanced AI Lab Topics

- Security: how a student could manipulate inputs
- Bias: sensitive variable check, fairness metric, mitigation
- Explainability: SHAP/LIME-style single prediction explanation

- Security: Some of the data used for failure prediction is provided by the students themselves. Could
a dishonest student lie about certain variables in order to obtain additional tutoring, even if they are
not among the most in need? Try to put yourself in the place of such a student and determine how
one could manipulate the inputs while changing as little information as possible. Is there a risk for the
OCR part?

- Bias: The dataset contains several variables. If sensitive variables are used, how do they affect the
model? Do they introduce unwanted bias? If so, you should evaluate this bias and apply techniques
to mitigate it while trying to maintain the best possible performance.
- Explainability: In the interest of transparency, a school may wish to explain to a student the factors
that made them eligible for special support. We ask you to provide an explanation based on which
and how much input variables influenced the decision. At the same time, and from a more scientific
perspective, a teacher interested in the OCR model would like to know which parts of the image are
used to predict the character. Could you provide such an explanation?



To respond to these aspects you have to submit a 5 page report supplement